In [ ]:
import os
import sys
import argparse
import torch
from pathlib import Path

sys.path.insert(0, os.path.dirname(__file__))

from cnns_backbones import CNNStem
from classifiers import GapBaseline, MambaClassifier, FlattenBaseline
from trainer import run_loso

MODEL_REGISTRY = {
    'gap': GapBaseline,
    'mamba': MambaClassifier,
    'flatten': FlattenBaseline,
}


def create_model_fn(model_type, dropout=0.2):
    backbone = CNNStem(dropout=dropout)
    model_cls = MODEL_REGISTRY[model_type]

    def model_fn():
        return model_cls(backbone)

    return model_fn


def main():
    parser = argparse.ArgumentParser(description='NanoSquiggle-AMR Training Sweep')
    parser.add_argument('--h5_path', required=True, help='Path to amr_features.h5')
    parser.add_argument('--model_type', required=True, choices=['gap', 'mamba', 'flatten'],
                        help='Model architecture to train')
    parser.add_argument('--epochs', type=int, default=50, help='Max epochs')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size')
    parser.add_argument('--lr', type=float, default=3e-4, help='Learning rate')
    parser.add_argument('--weight_decay', type=float, default=1e-4, help='Weight decay')
    parser.add_argument('--patience', type=int, default=10, help='Early stopping patience')
    parser.add_argument('--dropout', type=float, default=0.2, help='CNNStem dropout')
    parser.add_argument('--output_dir', type=str, default='training_output',
                        help='Output directory for results')
    parser.add_argument('--legacy', action='store_true',
                        help='Acknowledge flatten model is legacy (non-primary ablation)')
    args = parser.parse_args()

    if args.model_type == 'flatten' and not args.legacy:
        print("Warning: Flatten model has 17x more parameters than Mamba.")
        print("Results are not comparable. Use --legacy to acknowledge and run.")
        return

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Model: {args.model_type}, Epochs: {args.epochs}, Batch: {args.batch_size}, LR: {args.lr}")

    config = {
        'max_epochs': args.epochs,
        'batch_size': args.batch_size,
        'lr': args.lr,
        'weight_decay': args.weight_decay,
        'early_stop_patience': args.patience,
    }

    model_fn = create_model_fn(args.model_type, dropout=args.dropout)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    run_loso(model_fn, args.h5_path, args.model_type, config, device, str(output_dir))


if __name__ == '__main__':
    main()


NameError: name '__file__' is not defined